In [3]:
import fitz

In [4]:
pdf_path = "..\\documents\\spark.pdf"


In [17]:
doc= fitz.open(pdf_path)

with open("..\\extracted\\spark.txt","w+", encoding="utf-8") as f:
    for page_num,page in enumerate(doc):
        f.write(f"<slide {page_num}>")
        f.write(page.get_text())
doc.close()


In [18]:
import fitz # PyMuPDF library
import os
import io

def extract_images_from_pdf(pdf_path, output_folder="extracted_images"):
    """
    Extracts unique images from a PDF document using the PyMuPDF (fitz) library.

    Args:
        pdf_path (str): The file path to the input PDF document.
        output_folder (str): The directory where the extracted images will be saved.
    """
    
    # 1. Setup paths and check existence
    if not os.path.exists(pdf_path):
        print(f"ERROR: PDF file not found at: {pdf_path}")
        return

    os.makedirs(output_folder, exist_ok=True)
    
    doc = None
    
    try:
        # Open the PDF document
        doc = fitz.open(pdf_path)
        print(f"Successfully opened document: {pdf_path}")
        
        # Keep track of unique image references (based on xref) to avoid duplicates
        seen_images = set()
        image_count = 0
        
        # Loop through every page in the document
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            
            # The get_images() method returns a list of lists/tuples, where 
            # the first item in each inner list is the cross-reference (xref) number.
            image_list = page.get_images(full=True)
            
            for img_info in image_list:
                xref = img_info[0]
                
                # Check if we have already processed this image
                if xref in seen_images:
                    continue
                
                # 2. Extract the image data
                # doc.extract_image() returns a dictionary with the binary data and extension
                img_dict = doc.extract_image(xref)
                
                # Check if it's a valid image (sometimes non-image objects are returned)
                if not img_dict or 'image' not in img_dict:
                    continue
                    
                image_data = img_dict["image"]
                image_ext = img_dict["ext"]
                
                # 3. Save the image file
                # Create a unique filename based on the xref and page number
                filename = f"img_p{page_num+1}_xref{xref}.{image_ext}"
                output_path = os.path.join(output_folder, filename)
                
                with open(output_path, "wb") as img_file:
                    img_file.write(image_data)
                
                seen_images.add(xref)
                image_count += 1
                
                print(f"  Extracted image {image_count}: {filename}")

        print(f"\n--- Extraction Complete ---")
        print(f"Total unique images extracted: {image_count}")
        print(f"Images saved to directory: {output_folder}")
        
    except Exception as e:
        print(f"\nAn error occurred during image extraction: {e}")
    finally:
        if doc:
            doc.close()


# ----------------------------------------------------
# Example Usage: Replace with your actual document path
# ----------------------------------------------------
if __name__ == "__main__":
    # --- IMPORTANT ---
    # 1. Replace this placeholder path with the actual location of your PDF file.
    # 2. Ensure you have PyMuPDF installed: pip install PyMuPDF
    pdf_document_path = pdf_path
    
    # Create the output directory relative to the script
    output_directory = "pdf_images_output"
    
    extract_images_from_pdf(pdf_document_path, output_directory)


Successfully opened document: ..\documents\spark.pdf
  Extracted image 1: img_p2_xref19.jpeg
  Extracted image 2: img_p5_xref33.png
  Extracted image 3: img_p5_xref34.jpeg
  Extracted image 4: img_p6_xref37.png
  Extracted image 5: img_p15_xref65.jpeg
  Extracted image 6: img_p16_xref68.png

--- Extraction Complete ---
Total unique images extracted: 6
Images saved to directory: pdf_images_output


In [24]:
import fitz # PyMuPDF library
import os
import io

extracted_image_info=[]
def extract_images_from_pdf(pdf_path, output_folder="extracted_images"):
    """
    Extracts unique images from a PDF document, and captures the page number 
    where the image was first encountered.

    Args:
        pdf_path (str): The file path to the input PDF document.
        output_folder (str): The directory where the extracted images will be saved.
    """
    
    # 1. Setup paths and check existence
    if not os.path.exists(pdf_path):
        print(f"ERROR: PDF file not found at: {pdf_path}")
        return

    os.makedirs(output_folder, exist_ok=True)
    
    doc = None
    
    # We will use a list of dictionaries to store final results, including page number

    try:
        # Open the PDF document
        doc = fitz.open(pdf_path)
        print(f"Successfully opened document: {pdf_path}")
        
        # Keep track of unique image references (based on xref) to avoid duplicates
        seen_images = set()
        image_count = 0
        
        # Loop through every page in the document
        for page_num in range(len(doc)):
            # PDF page numbers are 0-indexed, so add 1 for user readability
            human_page_num = page_num + 1 
            page = doc.load_page(page_num)
            
            # The get_images() method returns a list of lists/tuples, where 
            # the first item in each inner list is the cross-reference (xref) number.
            image_list = page.get_images(full=True)
            
            for img_info in image_list:
                # Use .get() defensively, though xref is usually the first element
                xref = img_info[0]
                
                # Skip invalid xrefs
                if xref == 0:
                    continue
                
                # Check if we have already processed this image
                if xref in seen_images:
                    # Print location of duplicate instance (for context)
                    print(f"  [DUPLICATE] Image xref {xref} encountered again on Page {human_page_num}")
                    continue
                
                # 2. Extract the image data
                img_dict = doc.extract_image(xref)
                
                # Check if it's a valid image (sometimes non-image objects are returned)
                if not img_dict or 'image' not in img_dict:
                    continue
                    
                image_data = img_dict["image"]
                image_ext = img_dict["ext"]
                
                # 3. Save the image file
                # Use the page number where the image was *first* seen in the filename
                filename = f"img_p{human_page_num}_xref{xref}.{image_ext}"
                output_path = os.path.join(output_folder, filename)
                
                with open(output_path, "wb") as img_file:
                    img_file.write(image_data)
                
                # Record as seen and add to the final list
                seen_images.add(xref)
                image_count += 1
                
                extracted_image_info.append({
                    "filename": filename,
                    "xref": xref,
                    "page_number": human_page_num
                })
                
                print(f"  Extracted image {image_count}: {filename} (First found on Page {human_page_num})")


        print(f"\n--- Extraction Complete ---")
        print(f"Total unique images extracted: {image_count}")
        print(f"Images saved to directory: {output_folder}")
        
        # Optional: Print a clean summary list of extracted images and their page numbers
        print("\n--- Image Summary ---")
        for info in extracted_image_info:
            print(f"File: {info['filename']} | Page: {info['page_number']} | Xref: {info['xref']}")
        
    except Exception as e:
        print(f"\nAn error occurred during image extraction: {e}")
    finally:
        if doc:
            doc.close()


# ----------------------------------------------------
# Example Usage: Replace with your actual document path
# ----------------------------------------------------
if __name__ == "__main__":
    # NOTE: The variable 'pdf_path' used in the original snippet is undefined here.
    # I am setting a placeholder to keep the script runnable, please replace it.
    pdf_document_path = r"..\documents\spark.pdf"
    
    # Create the output directory relative to the script
    output_directory = "pdf_images_output"
    
    extract_images_from_pdf(pdf_document_path, output_directory)


Successfully opened document: ..\documents\spark.pdf
  Extracted image 1: img_p2_xref19.jpeg (First found on Page 2)
  Extracted image 2: img_p5_xref33.png (First found on Page 5)
  Extracted image 3: img_p5_xref34.jpeg (First found on Page 5)
  Extracted image 4: img_p6_xref37.png (First found on Page 6)
  Extracted image 5: img_p15_xref65.jpeg (First found on Page 15)
  Extracted image 6: img_p16_xref68.png (First found on Page 16)

--- Extraction Complete ---
Total unique images extracted: 6
Images saved to directory: pdf_images_output

--- Image Summary ---
File: img_p2_xref19.jpeg | Page: 2 | Xref: 19
File: img_p5_xref33.png | Page: 5 | Xref: 33
File: img_p5_xref34.jpeg | Page: 5 | Xref: 34
File: img_p6_xref37.png | Page: 6 | Xref: 37
File: img_p15_xref65.jpeg | Page: 15 | Xref: 65
File: img_p16_xref68.png | Page: 16 | Xref: 68


In [25]:
extracted_image_info

[{'filename': 'img_p2_xref19.jpeg', 'xref': 19, 'page_number': 2},
 {'filename': 'img_p5_xref33.png', 'xref': 33, 'page_number': 5},
 {'filename': 'img_p5_xref34.jpeg', 'xref': 34, 'page_number': 5},
 {'filename': 'img_p6_xref37.png', 'xref': 37, 'page_number': 6},
 {'filename': 'img_p15_xref65.jpeg', 'xref': 65, 'page_number': 15},
 {'filename': 'img_p16_xref68.png', 'xref': 68, 'page_number': 16}]

In [29]:
import re
import os
from typing import List, Dict, Any

In [30]:
# --- 1. LATEX TEMPLATE CONSTANTS ---

LATEX_TEMPLATE_START = r"""
\documentclass[12pt, openany]{book}
\usepackage{graphicx}
\usepackage{amsmath}
\usepackage{amssymb}
\usepackage{amsthm}
\usepackage{enumitem}
\usepackage{titlesec}
\usepackage{fancyhdr}
\usepackage{geometry}
\usepackage{xcolor}
\usepackage{color}
\usepackage{listings}
\usepackage{float}
\usepackage{hyperref}
\usepackage{booktabs}

% Page layout
\geometry{a4paper, margin=1in}
\pagestyle{fancy}
\fancyhf{}
\fancyhead[L]{\leftmark}
\fancyhead[R]{\thepage}
\renewcommand{\headrulewidth}{0.4pt}

% Chapter formatting
\titleformat{\chapter}[display]
{\normalfont\huge\bfseries}
{\chaptertitlename\ \thechapter}{20pt}{\Huge}

% Custom colors
\definecolor{codegray}{rgb}{0.5,0.5,0.5}
\definecolor{codepurple}{rgb}{0.58,0,0.82}
\definecolor{backcolour}{rgb}{0.95,0.95,0.92}  % Light gray background

% Listings configuration
\lstdefinestyle{mystyle}{
    backgroundcolor=\color{backcolour},
    commentstyle=\color{codegray},
    keywordstyle=\color{codepurple},
    numberstyle=\tiny\color{codegray},
    stringstyle=\color{codepurple},
    basicstyle=\ttfamily\footnotesize\color{black},  % Explicitly set text color
    breakatwhitespace=false,
    breaklines=true,
    captionpos=b,
    keepspaces=true,
    numbers=left,
    numbersep=5pt,
    showspaces=false,
    showstringspaces=false,
    showtabs=false,
    tabsize=2
}
\lstset{style=mystyle}

% Remove footers with professor name
\fancyfoot{}
\renewcommand{\rmdefault}{qpl}

\begin{document}

\setcounter{page}{45}
\setcounter{chapter}{4}
\chapter{Big Data Frameworks - RDD}
"""

LATEX_TEMPLATE_END = r"""
\end{document}
"""

In [31]:
# --- 2. HELPER FUNCTIONS ---

def group_images_by_page(image_info: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    """Groups image metadata by their 1-indexed page number."""
    page_to_images = {}
    for img in image_info:
        page_num = img['page_number']
        if page_num not in page_to_images:
            page_to_images[page_num] = []
        page_to_images[page_num].append(img)
    return page_to_images

def parse_and_generate_latex(raw_text: str, image_info: List[Dict[str, Any]], output_filename: str = "generated_textbook_chapter.tex"):
    """
    Parses the slide text and image info to generate the final LaTeX document.
    """
    
    image_map = group_images_by_page(image_info)
    
    # Split the text by slide tags, keeping the tag itself (e.g., <slide 0>, <slide 1>)
    # This results in [content_before_tag1, tag1, content_after_tag1, tag2, ...]
    slide_chunks = re.split(r'(<slide\s*\d+>)', raw_text.strip())
    
    # Discard the first element if it's empty (text before the first tag)
    if slide_chunks and not slide_chunks[0].strip():
        slide_chunks = slide_chunks[1:]

    latex_content_parts = []
    current_section_title = None
    
    # Iterate over tag and content pairs (tag at i, content at i+1)
    for i in range(0, len(slide_chunks), 2):
        if i + 1 >= len(slide_chunks):
            break

        tag = slide_chunks[i].strip() # e.g., <slide 1>
        content_block = slide_chunks[i+1].strip()
        
        # 1. Extract Slide Number (0-indexed)
        match = re.search(r'<slide\s*(\d+)>', tag)
        if not match:
            continue
            
        # Slide number is 0-indexed, image page_number is 1-indexed
        slide_num_0_indexed = int(match.group(1))
        page_num_1_indexed = slide_num_0_indexed + 1
        
        # 2. Extract Title and Body
        lines = content_block.splitlines()
        
        # The first non-empty line after the tag is the HEADING
        heading_line = lines[0].strip()
        
        # Body is the remaining lines
        body_lines = [line.strip() for line in lines[1:] if line.strip()]
        
        # 3. Process Body Content (Bullets and cleanup)
        processed_body = []
        is_itemize_block = False
        
        for line in body_lines:
            # Cleanup stray slide/page numbers (e.g., ' 2' or ' 3') at line ends
            line = re.sub(r'\s*\d+$', '', line).strip()
            
            if not line:
                continue
            
            # Convert list items (• or *) to LaTeX \item
            if line.startswith('•') or line.startswith('*'):
                processed_body.append(r'\item ' + line[1:].strip())
                is_itemize_block = True
            else:
                # Add as a new paragraph (with LaTeX newline)
                # Ensure text is escaped minimally
                escaped_line = line.replace('&', r'\&').replace('%', r'\%').replace('_', r'\_')
                processed_body.append('\n' + escaped_line + '\n')
        
        body_content = "\n".join(processed_body)
        
        # 4. Determine New Section Logic
        new_section_title = heading_line
        
        if new_section_title != current_section_title:
            # New unique section title found
            
            # Start new section
            latex_content_parts.append(f'\n\\section{{{new_section_title}}}')
            current_section_title = new_section_title
        
        # 5. Add Body Content
        if is_itemize_block:
            # Wrap all the list items in the itemize environment
            latex_content_parts.append('\n\\begin{itemize}[leftmargin=*]')
            latex_content_parts.append(body_content)
            latex_content_parts.append('\\end{itemize}\n')
        else:
            # If no list items, just add the content as a paragraph
            latex_content_parts.append('\n' + body_content + '\n')
        
        # 6. Insert Images (Matching slide number to image page_number)
        images_to_insert = image_map.get(page_num_1_indexed, [])
        if images_to_insert:
            for img in images_to_insert:
                filename = img['filename']
                # The path for \includegraphics assumes the image files are accessible to the compiler
                # You may need to adjust the path (e.g., 'pdf_images_output/') based on your setup.
                latex_content_parts.append(
                    f'\n\\begin{{figure}}[H]\n'
                    f'\\centering\n'
                    # Use a placeholder width; adjust as necessary
                    f'\\includegraphics[width=0.8\\textwidth]{{pdf_images_output/{filename}}}\n' 
                    f'\\caption{{Diagram related to: {new_section_title}}}\n'
                    f'\\end{{figure}}\n'
                )

    # 7. Final assembly and file generation
    full_latex_document = (
        LATEX_TEMPLATE_START + 
        "".join(latex_content_parts) + 
        LATEX_TEMPLATE_END
    )

    try:
        # Write to a file
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(full_latex_document)
        print(f"\n\n--- SUCCESS ---")
        print(f"✅ Generated LaTeX file: {output_filename}")
        print(f"1. Ensure your extracted image files are in the 'pdf_images_output/' directory.")
        print(f"2. Compile with pdflatex.")
    except Exception as e:
        print(f"❌ Error writing to file: {e}")

In [32]:
# --- 3. INPUT DATA ---

# --- IMPORTANT: RAW_INPUT_TEXT is now read from the file path. ---

FILE_PATH = os.path.join("..", "extracted", "spark.txt")
RAW_INPUT_TEXT = ""

try:
    with open(FILE_PATH, "r", encoding="utf-8") as f:
        RAW_INPUT_TEXT = f.read()
    print(f"Successfully read input text from: {FILE_PATH}")
except FileNotFoundError:
    print(f"File not found at {FILE_PATH}. Using hardcoded sample text for execution.")
    # Fallback content to ensure the script can run if the file doesn't exist
    RAW_INPUT_TEXT = """
<slide 0>CSE6001
BIG DATA FRAMEWORKS
Module - 5
RDD
Prof. Lokeshkumar R
<slide 1>Spark Memory Management 
2
• Memory utilization is essential in Spark 
(Caching)
• Spark process is a JVM process
<slide 2>Spark Programming Model
• High-Level coding to build a workflow (Scala)
• Code compiles to distributed parallel operations
• Two Abstraction Units
• RDDs: Resilient Distributed Datasets 
• Parallel Operations 
3
<slide 3>RDD: Concept
• Collection of objects (records) that act as one unit
• Stored in main memory or disk
• Parallel operations built on top of them
• Have fault tolerance without replication (lineage)
4
<slide 4>RDD: Concept
• RDD is read-only
• Distributed either in main memory or disk (automatically 
decided) 
5
<slide 5>RDD vs. Traditional Shared Memory
"""
except Exception as e:
    print(f"An unexpected error occurred while reading the file: {e}. Cannot generate LaTeX.")
    RAW_INPUT_TEXT = ""


# This dictionary is assumed to be generated by a previous step (like extract_pdf_images.py)
EXTRACTED_IMAGE_INFO = extracted_image_info

# --- 4. EXECUTION ---
if __name__ == "__main__":
    parse_and_generate_latex(RAW_INPUT_TEXT, EXTRACTED_IMAGE_INFO)


Successfully read input text from: ..\extracted\spark.txt


--- SUCCESS ---
✅ Generated LaTeX file: generated_textbook_chapter.tex
1. Ensure your extracted image files are in the 'pdf_images_output/' directory.
2. Compile with pdflatex.
